# Supply Scheduling Engine

This notebook implements an intelligent scheduling system that:

## Data Flow
```
INPUT:
  ├── output/predictions/daily_predictions.parquet  # Predicted passenger KMs
  ├── data/service_master.csv                       # Service schedule master
  └── data/processed/ops_daily_service_gold.parquet # Historical service operations
       ↓
PROCESSING:
  └── Policy Engine: Adjust trips based on predicted demand
       ↓
OUTPUT:
  └── output/dynamic_schedule/YYYY-MM-DD/
      ├── schedule_{depot}_{date}.xlsx    # Suggested schedule per depot
      ├── summary_{depot}_{date}.json     # Summary metrics
      └── consolidated_summary.xlsx       # All depots combined
```

## Actions
- **INCREASE**: Add trips to meet higher demand
- **DECREASE**: Reduce trips when demand is lower
- **STOP**: Stop service when demand is very low
- **NO_CHANGE**: Keep current schedule

## 1. Setup & Configuration

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import pandas as pd
import numpy as np
from datetime import time, datetime, date, timedelta
from collections import defaultdict
from pathlib import Path
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

print("Imports completed")

Imports completed


In [2]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================

# Project root - find dynamically by looking for marker directories
def find_project_root():
    """Find project root by looking for data/ or model/ directory."""
    candidates = [
        Path("..").resolve(),           # If running from notebooks/
        Path(".").resolve(),            # If running from project root
        Path(__file__).parent.parent if '__file__' in dir() else None,
    ]
    
    for candidate in candidates:
        if candidate and (candidate / "data").exists() and (candidate / "model").exists():
            return candidate
    
    # Fallback: explicit path
    fallback = Path("/home/sr/agnigent/ai_agents/dynamic_scheduling")
    if fallback.exists():
        return fallback
    
    raise RuntimeError("Could not find project root. Expected data/ and model/ directories.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "dynamic_schedule"

# Input files (master data)
SERVICE_MASTER_PATH = DATA_DIR / "master" / "service_master.csv"  # Schedule master

# Service-level operations: prefer parquet (GOLD), fallback to CSV (RAW)
SERVICE_OPS_GOLD_PATH = DATA_DIR / "processed" / "ops_daily_service_gold.parquet"
SERVICE_OPS_RAW_PATH = DATA_DIR / "raw" / "ops_daily_service_master.csv"

# Predictions file (from demand_prediction.ipynb)
PREDICTIONS_DIR = PROJECT_ROOT / "output" / "predictions"
PREDICTIONS_FILE = PREDICTIONS_DIR / "daily_predictions.parquet"

# Verify paths
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"\nInput files:")
print(f"  SERVICE_MASTER: {SERVICE_MASTER_PATH} (exists: {SERVICE_MASTER_PATH.exists()})")
print(f"  SERVICE_OPS (GOLD): {SERVICE_OPS_GOLD_PATH} (exists: {SERVICE_OPS_GOLD_PATH.exists()})")
print(f"  SERVICE_OPS (RAW): {SERVICE_OPS_RAW_PATH} (exists: {SERVICE_OPS_RAW_PATH.exists()})")
print(f"  PREDICTIONS: {PREDICTIONS_FILE} (exists: {PREDICTIONS_FILE.exists()})")
print(f"\nOutput:")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")

PROJECT_ROOT: /home/sr/agnigent/ai_agents/dynamic_scheduling

Input files:
  SERVICE_MASTER: /home/sr/agnigent/ai_agents/dynamic_scheduling/data/master/service_master.csv (exists: True)
  SERVICE_OPS (GOLD): /home/sr/agnigent/ai_agents/dynamic_scheduling/data/processed/ops_daily_service_gold.parquet (exists: True)
  SERVICE_OPS (RAW): /home/sr/agnigent/ai_agents/dynamic_scheduling/data/raw/ops_daily_service_master.csv (exists: True)
  PREDICTIONS: /home/sr/agnigent/ai_agents/dynamic_scheduling/output/predictions/daily_predictions.parquet (exists: True)

Output:
  OUTPUT_DIR: /home/sr/agnigent/ai_agents/dynamic_scheduling/output/dynamic_schedule


## 2. Load Data

In [3]:
# =============================================================================
# LOAD SERVICE MASTER (Schedule Master)
# =============================================================================

service_master_raw = pd.read_csv(SERVICE_MASTER_PATH)
print(f"Service master loaded: {service_master_raw.shape}")
print(f"Columns: {service_master_raw.columns.tolist()}")

# Check depot column
if 'depot' in service_master_raw.columns:
    print(f"\nDepots: {service_master_raw['depot'].nunique()} unique")
    print(f"  {service_master_raw['depot'].unique().tolist()}")
else:
    print("WARNING: No 'depot' column found in service master")

Service master loaded: (136, 10)
Columns: ['depot', 'service_number', 'product', 'avg_seats_per_bus', 'route', 'dep_time', 'arr_time', 'planned_kms', 'planned_trips', 'planned_per_trip']

Depots: 1 unique
  ['WARANGAL-I', nan]


In [4]:
# =============================================================================
# LOAD SERVICE-LEVEL OPERATIONS
# Prefer GOLD parquet, fallback to RAW CSV
# =============================================================================

if SERVICE_OPS_GOLD_PATH.exists():
    print(f"Loading GOLD parquet: {SERVICE_OPS_GOLD_PATH}")
    daily_ops_raw = pd.read_parquet(SERVICE_OPS_GOLD_PATH)
    data_source = "GOLD"
elif SERVICE_OPS_RAW_PATH.exists():
    print(f"GOLD not found, loading RAW CSV: {SERVICE_OPS_RAW_PATH}")
    daily_ops_raw = pd.read_csv(SERVICE_OPS_RAW_PATH)
    data_source = "RAW"
else:
    raise FileNotFoundError("No service operations data found!")

print(f"\nDaily ops loaded ({data_source}): {daily_ops_raw.shape}")
print(f"Columns: {daily_ops_raw.columns.tolist()}")

if 'depot' in daily_ops_raw.columns:
    print(f"\nDepots in ops: {daily_ops_raw['depot'].nunique()} unique")

Loading GOLD parquet: /home/sr/agnigent/ai_agents/dynamic_scheduling/data/processed/ops_daily_service_gold.parquet

Daily ops loaded (GOLD): (2720, 17)
Columns: ['depot', 'date', 'service_number', 'actual_kms', 'actual_trips', 'seat_kms', 'passenger_kms', 'occupancy_ratio', 'telugu_thithi', 'telugu_paksham', 'marriage_day', 'telugu_month', 'moudyami_day', 'fes_hol_code', 'Holiday_Festival', 'fes_hol_category', 'is_fes_hol']

Depots in ops: 1 unique


In [5]:
# =============================================================================
# DATA CLEANING
# =============================================================================

def clean_service_master(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and prepare service master data."""
    df = df.copy()
    
    if 'depot' not in df.columns:
        df['depot'] = 'DEFAULT'
    
    df['depot'] = df['depot'].astype(str).str.strip()
    df['service_number'] = df['service_number'].astype(str).str.strip()
    df['planned_trips'] = pd.to_numeric(df['planned_trips'], errors='coerce').fillna(0).astype(int)
    df['planned_kms'] = pd.to_numeric(df['planned_kms'], errors='coerce').fillna(0)
    df['avg_seats_per_bus'] = pd.to_numeric(df['avg_seats_per_bus'], errors='coerce').fillna(45)
    
    if 'km_per_trip' not in df.columns or df['km_per_trip'].isna().all():
        df['km_per_trip'] = df['planned_kms'] / df['planned_trips'].replace(0, np.nan)
    df['km_per_trip'] = pd.to_numeric(df['km_per_trip'], errors='coerce').fillna(0)
    
    return df


def clean_daily_ops(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and prepare daily operations data."""
    df = df.copy()
    
    if 'depot' not in df.columns:
        df['depot'] = 'DEFAULT'
    
    df['depot'] = df['depot'].astype(str).str.strip()
    df['service_number'] = df['service_number'].astype(str).str.strip()
    
    # Handle date column
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Numeric columns
    num_cols = ['actual_kms', 'actual_trips', 'seat_kms', 'passenger_kms', 'occupancy_ratio']
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if 'actual_trips' in df.columns:
        df['actual_trips'] = df['actual_trips'].fillna(0)
    
    return df


# Apply cleaning
service_master = clean_service_master(service_master_raw)
daily_ops = clean_daily_ops(daily_ops_raw)

print("Data cleaning complete")
print(f"Service master: {len(service_master)} services across {service_master['depot'].nunique()} depot(s)")
print(f"Daily ops: {len(daily_ops)} records")

Data cleaning complete
Service master: 136 services across 1 depot(s)
Daily ops: 2720 records


## 3. Policy Configuration

In [6]:
# =============================================================================
# POLICY CONFIGURATION
# All business rules and thresholds in one place
# =============================================================================

POLICY = {
    # Target occupancy ratio for scheduling
    "target_or": 0.75,
    
    # Tolerance: if delta KM is within ±X% of planned, do nothing
    "tolerance_pct": 0.03,
    
    # Lookback window for calculating recent occupancy ratio
    "lookback_days": 7,
    
    # Maximum trips to add/remove per service in one day
    "max_trip_change_per_service": 1,
    
    # Minimum trips a service must maintain (0 = can stop completely)
    "min_trips_per_service": 0,
    
    # Occupancy ratio thresholds
    "underutilized_or": 0.65,   # Below this: don't add trips
    "overloaded_or": 0.95,      # Above this: prioritize adding
    "stop_or": 0.45,            # Below this: can stop service
    
    # Peak hour windows
    "morning_peak": (time(6, 0), time(10, 0)),
    "evening_peak": (time(16, 30), time(20, 30)),
    
    # Peak service preferences
    "prefer_peak_when_adding": True,
    "protect_peak_when_cutting": True,
    
    # Route grouping
    "max_changes_per_route": 2,
    "route_column": "route",
    
    # Protected services (never cut)
    "protected_services": set(),
    
    # Global cap per depot per day
    "max_services_changed": 25,
    
    # Prefer shorter km-per-trip when adding
    "prefer_short_kmpt_when_adding": True,
}

print("Policy Configuration:")
print("-" * 50)
for key, value in POLICY.items():
    if key not in ['protected_services']:
        print(f"  {key}: {value}")

Policy Configuration:
--------------------------------------------------
  target_or: 0.75
  tolerance_pct: 0.03
  lookback_days: 7
  max_trip_change_per_service: 1
  min_trips_per_service: 0
  underutilized_or: 0.65
  overloaded_or: 0.95
  stop_or: 0.45
  morning_peak: (datetime.time(6, 0), datetime.time(10, 0))
  evening_peak: (datetime.time(16, 30), datetime.time(20, 30))
  prefer_peak_when_adding: True
  protect_peak_when_cutting: True
  max_changes_per_route: 2
  route_column: route
  max_services_changed: 25
  prefer_short_kmpt_when_adding: True


## 4. Helper Functions

In [7]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def parse_time_safe(x) -> time | None:
    """Safely parse time from various formats."""
    if pd.isna(x):
        return None
    if isinstance(x, time):
        return x
    try:
        return pd.to_datetime(str(x)).time()
    except Exception:
        return None


def is_peak_hour(dep_time_val, policy: dict) -> bool:
    """Check if departure time falls within peak hours."""
    t = parse_time_safe(dep_time_val)
    if t is None:
        return False
    m1, m2 = policy["morning_peak"]
    e1, e2 = policy["evening_peak"]
    return (m1 <= t <= m2) or (e1 <= t <= e2)


def compute_recent_or(
    daily_ops: pd.DataFrame,
    depot: str,
    target_date: date,
    lookback_days: int
) -> pd.Series:
    """Compute average occupancy ratio per service over lookback window."""
    depot_ops = daily_ops[daily_ops['depot'] == depot].copy()
    
    if 'date' in depot_ops.columns and pd.api.types.is_datetime64_any_dtype(depot_ops['date']):
        # Filter to lookback window
        end_date = pd.Timestamp(target_date)
        start_date = end_date - pd.Timedelta(days=lookback_days)
        depot_ops = depot_ops[(depot_ops['date'] >= start_date) & (depot_ops['date'] < end_date)]
    else:
        # Use all available data
        pass
    
    if 'occupancy_ratio' in depot_ops.columns:
        return depot_ops.groupby('service_number')['occupancy_ratio'].mean()
    return pd.Series(dtype=float)


def compute_depot_planned_kms(service_master: pd.DataFrame, depot: str) -> float:
    """Compute total planned KMs for a depot."""
    depot_services = service_master[service_master['depot'] == depot]
    return float((depot_services['planned_trips'] * depot_services['km_per_trip']).sum())


def compute_target_kms(
    predicted_pkm: float,
    avg_seats: float,
    target_or: float
) -> float:
    """Compute target operational KMs based on predicted passenger-KMs."""
    if not np.isfinite(avg_seats) or avg_seats <= 0:
        avg_seats = 45
    target_seat_km = predicted_pkm / max(target_or, 1e-6)
    return float(target_seat_km / avg_seats)


print("Helper functions defined")

Helper functions defined


## 5. Policy Engine

In [8]:
# =============================================================================
# POLICY ENGINE - Core scheduling logic
# =============================================================================

def run_policy_engine(
    service_master: pd.DataFrame,
    daily_ops: pd.DataFrame,
    depot: str,
    target_date: date,
    predicted_depot_pkm: float,
    policy: dict
) -> tuple[pd.DataFrame, dict]:
    """
    Run the scheduling policy engine for a single depot.
    
    Returns:
        (schedule_df, summary_dict)
    """
    # Filter to depot
    base = service_master[service_master['depot'] == depot].copy()
    
    if len(base) == 0:
        return pd.DataFrame(), {"error": f"No services found for depot {depot}"}
    
    # 1) Compute recent OR per service
    recent_or = compute_recent_or(daily_ops, depot, target_date, policy['lookback_days'])
    or_col = f"avg_or_last_{policy['lookback_days']}d"
    base = base.merge(recent_or.rename(or_col), on='service_number', how='left')
    base[or_col] = base[or_col].fillna(0)
    
    # 2) Add peak and protected flags
    base['is_peak'] = base['dep_time'].apply(lambda x: is_peak_hour(x, policy))
    base['is_protected'] = base['service_number'].isin(policy['protected_services'])
    
    route_col = policy['route_column']
    if route_col not in base.columns:
        base[route_col] = 'UNKNOWN'
    base[route_col] = base[route_col].astype(str).fillna('UNKNOWN')
    
    # 3) Compute planned vs target KMs
    planned_kms = compute_depot_planned_kms(service_master, depot)
    avg_seats = base['avg_seats_per_bus'].replace(0, np.nan).mean()
    target_kms = compute_target_kms(predicted_depot_pkm, avg_seats, policy['target_or'])
    delta_kms = target_kms - planned_kms
    
    tol_kms = policy['tolerance_pct'] * max(planned_kms, 1.0)
    should_act = abs(delta_kms) > tol_kms
    
    # 4) Initialize schedule columns
    base['suggested_trips'] = base['planned_trips'].copy()
    base['action'] = 'NO_CHANGE'
    base['reason'] = 'Within tolerance'
    
    total_changed = 0
    route_changes = defaultdict(int)
    
    def can_change_route(route_val: str) -> bool:
        return route_changes[route_val] < policy['max_changes_per_route']
    
    def register_change(route_val: str):
        nonlocal total_changed
        route_changes[route_val] += 1
        total_changed += 1
    
    def reached_cap() -> bool:
        return (policy['max_services_changed'] is not None and 
                total_changed >= policy['max_services_changed'])
    
    # 5) Apply policy decisions
    if should_act:
        if delta_kms > 0:
            # === ADD TRIPS ===
            remaining = delta_kms
            add_df = base[base[or_col] >= policy['underutilized_or']].copy()
            
            sort_cols, sort_asc = [], []
            if policy['prefer_peak_when_adding']:
                sort_cols.append('is_peak')
                sort_asc.append(False)
            sort_cols.append(or_col)
            sort_asc.append(False)
            if policy['prefer_short_kmpt_when_adding']:
                sort_cols.append('km_per_trip')
                sort_asc.append(True)
            
            add_df = add_df.sort_values(sort_cols, ascending=sort_asc)
            
            for idx in add_df.index:
                if remaining <= 0 or reached_cap():
                    break
                route_val = base.loc[idx, route_col]
                if not can_change_route(route_val):
                    continue
                
                add_n = policy['max_trip_change_per_service']
                km_gain = float(base.loc[idx, 'km_per_trip']) * add_n
                if km_gain <= 0:
                    continue
                
                base.loc[idx, 'suggested_trips'] += add_n
                base.loc[idx, 'action'] = 'INCREASE'
                base.loc[idx, 'reason'] = f"Add {add_n} trip(s), OR={float(base.loc[idx,or_col]):.2f}"
                remaining -= km_gain
                register_change(route_val)
        else:
            # === CUT / STOP TRIPS ===
            need_remove = -delta_kms
            cut_df = base[~base['is_protected']].copy()
            
            if policy['protect_peak_when_cutting']:
                cut_first = cut_df[~cut_df['is_peak']].copy()
                cut_later = cut_df[cut_df['is_peak']].copy()
            else:
                cut_first = cut_df
                cut_later = pd.DataFrame()
            
            cut_first = cut_first.sort_values([or_col, 'km_per_trip'], ascending=[True, False])
            cut_later = cut_later.sort_values([or_col, 'km_per_trip'], ascending=[True, False])
            
            def try_cut(df_part: pd.DataFrame):
                nonlocal need_remove
                for idx in df_part.index:
                    if need_remove <= 0 or reached_cap():
                        break
                    route_val = base.loc[idx, route_col]
                    if not can_change_route(route_val):
                        continue
                    
                    or_i = float(base.loc[idx, or_col])
                    current = int(base.loc[idx, 'suggested_trips'])
                    
                    if current <= policy['min_trips_per_service']:
                        continue
                    
                    if or_i < policy['stop_or'] and policy['min_trips_per_service'] == 0:
                        remove_n = current
                        base.loc[idx, 'suggested_trips'] = 0
                        base.loc[idx, 'action'] = 'STOP'
                        base.loc[idx, 'reason'] = f"Stop service, very low OR={or_i:.2f}"
                    else:
                        remove_n = min(
                            policy['max_trip_change_per_service'],
                            current - policy['min_trips_per_service']
                        )
                        if remove_n <= 0:
                            continue
                        base.loc[idx, 'suggested_trips'] = current - remove_n
                        base.loc[idx, 'action'] = 'DECREASE'
                        base.loc[idx, 'reason'] = f"Cut {remove_n} trip(s), OR={or_i:.2f}"
                    
                    km_saved = float(base.loc[idx, 'km_per_trip']) * remove_n
                    need_remove -= km_saved
                    register_change(route_val)
            
            try_cut(cut_first)
            if need_remove > 0:
                try_cut(cut_later)
    
    # 6) Compute impacts
    base['planned_kms_day'] = (base['planned_trips'] * base['km_per_trip']).round(2)
    base['suggested_kms_day'] = (base['suggested_trips'] * base['km_per_trip']).round(2)
    base['trip_change'] = (base['suggested_trips'] - base['planned_trips']).astype(int)
    base['kms_change'] = (base['suggested_kms_day'] - base['planned_kms_day']).round(2)
    
    # 7) Format output
    out_cols = [
        'service_number', route_col, 'product', 'dep_time',
        'is_peak', 'is_protected',
        'planned_trips', 'suggested_trips', 'trip_change',
        'km_per_trip', 'planned_kms_day', 'suggested_kms_day', 'kms_change',
        or_col, 'action', 'reason'
    ]
    out_cols = [c for c in out_cols if c in base.columns]
    out = base[out_cols].copy()
    
    action_order = pd.Categorical(
        out['action'],
        categories=['STOP', 'DECREASE', 'INCREASE', 'NO_CHANGE'],
        ordered=True
    )
    out['_action_rank'] = action_order
    out = out.sort_values(['_action_rank', 'kms_change'], ascending=[True, False])
    out = out.drop(columns='_action_rank').reset_index(drop=True)
    
    # 8) Summary
    summary = {
        'depot': depot,
        'target_date': str(target_date),
        'predicted_depot_pkm': float(predicted_depot_pkm),
        'policy_target_or': float(policy['target_or']),
        'depot_planned_kms': float(planned_kms),
        'depot_target_kms': float(target_kms),
        'delta_kms': float(delta_kms),
        'tolerance_kms': float(tol_kms),
        'action_taken': bool(should_act),
        'total_kms_change': float(out['kms_change'].sum()),
        'count_increase': int((out['action'] == 'INCREASE').sum()),
        'count_decrease': int((out['action'] == 'DECREASE').sum()),
        'count_stop': int((out['action'] == 'STOP').sum()),
        'count_no_change': int((out['action'] == 'NO_CHANGE').sum()),
        'total_services_changed': total_changed,
    }
    
    return out, summary


print("Policy engine defined")

Policy engine defined


## 6. Multi-Depot Runner

In [9]:
# =============================================================================
# MULTI-DEPOT SCHEDULING RUNNER
# =============================================================================

def run_all_depots(
    service_master: pd.DataFrame,
    daily_ops: pd.DataFrame,
    target_date: date,
    depot_predictions: dict,
    policy: dict,
    output_dir: Path,
) -> tuple[dict, pd.DataFrame]:
    """
    Run scheduling for all depots and save outputs.
    """
    date_str = target_date.strftime('%Y-%m-%d')
    date_output_dir = output_dir / date_str
    date_output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Output directory: {date_output_dir}")
    print("=" * 60)
    
    all_summaries = {}
    all_schedules = []
    
    for depot, predicted_pkm in depot_predictions.items():
        print(f"\nProcessing depot: {depot}")
        print(f"  Predicted PKM: {predicted_pkm:,.0f}")
        
        schedule_df, summary = run_policy_engine(
            service_master=service_master,
            daily_ops=daily_ops,
            depot=depot,
            target_date=target_date,
            predicted_depot_pkm=predicted_pkm,
            policy=policy
        )
        
        if 'error' in summary:
            print(f"  ERROR: {summary['error']}")
            continue
        
        schedule_df['depot'] = depot
        all_schedules.append(schedule_df)
        all_summaries[depot] = summary
        
        print(f"  Planned KMs: {summary['depot_planned_kms']:,.0f}")
        print(f"  Target KMs: {summary['depot_target_kms']:,.0f}")
        print(f"  Delta: {summary['delta_kms']:+,.0f}")
        print(f"  Actions: +{summary['count_increase']} / -{summary['count_decrease']} / STOP:{summary['count_stop']}")
        
        depot_safe = depot.replace(' ', '_').replace('/', '_')
        
        schedule_path = date_output_dir / f"schedule_{depot_safe}_{date_str}.xlsx"
        schedule_df.to_excel(schedule_path, index=False)
        print(f"  Saved: {schedule_path.name}")
        
        summary_path = date_output_dir / f"summary_{depot_safe}_{date_str}.json"
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
    
    if all_schedules:
        consolidated_df = pd.concat(all_schedules, ignore_index=True)
        
        consolidated_path = date_output_dir / f"consolidated_schedule_{date_str}.xlsx"
        consolidated_df.to_excel(consolidated_path, index=False)
        print(f"\nSaved consolidated: {consolidated_path.name}")
        
        consolidated_summary_path = date_output_dir / f"consolidated_summary_{date_str}.json"
        with open(consolidated_summary_path, 'w') as f:
            json.dump(all_summaries, f, indent=2, default=str)
    else:
        consolidated_df = pd.DataFrame()
    
    return all_summaries, consolidated_df


print("Multi-depot runner defined")

Multi-depot runner defined


## 7. Load Predictions

In [10]:
# =============================================================================
# LOAD PREDICTIONS FROM PREDICTIONS FILE
# =============================================================================

def load_predictions_for_scheduling(
    predictions_file: Path,
    target_date: date = None,
) -> dict:
    """
    Load predictions from the predictions parquet file.
    
    Parameters:
        predictions_file: Path to daily_predictions.parquet
        target_date: Date to get predictions for. If None, uses tomorrow.
    
    Returns:
        Dict of {depot: predicted_passenger_kms}
    """
    if not predictions_file.exists():
        print(f"Predictions file not found: {predictions_file}")
        return {}
    
    df = pd.read_parquet(predictions_file)
    df["prediction_date"] = pd.to_datetime(df["prediction_date"])
    
    if target_date is None:
        target_date = date.today() + timedelta(days=1)
    
    print(f"Looking for predictions for: {target_date}")
    
    date_mask = df["prediction_date"].dt.date == target_date
    target_predictions = df[date_mask]
    
    if len(target_predictions) == 0:
        print(f"No predictions found for {target_date}")
        available = sorted(df['prediction_date'].dt.date.unique())
        print(f"Available dates: {available[-5:] if len(available) > 5 else available}")
        return {}
    
    depot_predictions = {}
    for _, row in target_predictions.iterrows():
        depot = row["depot"]
        
        # Use passenger_kms if available, otherwise convert from passengers
        if "predicted_passenger_kms" in row and pd.notna(row["predicted_passenger_kms"]):
            depot_predictions[depot] = row["predicted_passenger_kms"]
        elif "estimated_kms" in row and pd.notna(row["estimated_kms"]):
            # Use estimated_kms as proxy for passenger_kms
            depot_predictions[depot] = row["estimated_kms"] * 45 * 0.75  # seat_kms * OR
        else:
            # Convert passengers to passenger-kms
            predicted_passengers = row.get("predicted_passengers", 0)
            AVG_KM_PER_PASSENGER = 25
            depot_predictions[depot] = predicted_passengers * AVG_KM_PER_PASSENGER
    
    return depot_predictions


print("Load predictions function defined")

Load predictions function defined


In [11]:
# =============================================================================
# GET TARGET DATE AND PREDICTIONS
# =============================================================================

# Target date: tomorrow by default
TARGET_DATE = date.today() + timedelta(days=1)

# Or specify a specific date:
# TARGET_DATE = date(2026, 1, 30)

print(f"Scheduling for date: {TARGET_DATE}")
print("=" * 60)

# Load predictions from file
DEPOT_PREDICTIONS = load_predictions_for_scheduling(
    predictions_file=PREDICTIONS_FILE,
    target_date=TARGET_DATE,
)

if DEPOT_PREDICTIONS:
    print("\nPredictions loaded:")
    for depot, pkm in DEPOT_PREDICTIONS.items():
        print(f"  {depot}: {pkm:,.0f} PKM")
else:
    print("\nNo predictions found. Using historical averages...")
    
    all_depots = service_master['depot'].unique().tolist()
    
    if 'passenger_kms' in daily_ops.columns:
        historical_pkm = (
            daily_ops
            .groupby(['depot', 'date'])['passenger_kms']
            .sum()
            .groupby('depot')
            .mean()
        )
        DEPOT_PREDICTIONS = historical_pkm.to_dict()
        print("Historical averages:")
        for depot, pkm in DEPOT_PREDICTIONS.items():
            print(f"  {depot}: {pkm:,.0f} PKM")
    else:
        DEPOT_PREDICTIONS = {depot: 2_000_000 for depot in all_depots}
        print(f"Using default 2,000,000 PKM for all depots")

Scheduling for date: 2026-01-30
Looking for predictions for: 2026-01-30

Predictions loaded:
  CONTONMENT: 5,400,341 PKM
  KARIMNAGAR-I: 2,233,441 PKM
  NIZAMABAD-I: 2,267,471 PKM
  WARANGAL-I: 2,061,263 PKM


## 8. Run Scheduling

In [12]:
# =============================================================================
# RUN SCHEDULING FOR ALL DEPOTS
# =============================================================================

all_summaries, consolidated_schedule = run_all_depots(
    service_master=service_master,
    daily_ops=daily_ops,
    target_date=TARGET_DATE,
    depot_predictions=DEPOT_PREDICTIONS,
    policy=POLICY,
    output_dir=OUTPUT_DIR,
)

print("\n" + "=" * 60)
print("SCHEDULING COMPLETE")
print("=" * 60)

Output directory: /home/sr/agnigent/ai_agents/dynamic_scheduling/output/dynamic_schedule/2026-01-30

Processing depot: CONTONMENT
  Predicted PKM: 5,400,341
  ERROR: No services found for depot CONTONMENT

Processing depot: KARIMNAGAR-I
  Predicted PKM: 2,233,441
  ERROR: No services found for depot KARIMNAGAR-I

Processing depot: NIZAMABAD-I
  Predicted PKM: 2,267,471
  ERROR: No services found for depot NIZAMABAD-I

Processing depot: WARANGAL-I
  Predicted PKM: 2,061,263
  Planned KMs: 69,696
  Target KMs: 61,074
  Delta: -8,622
  Actions: +0 / -0 / STOP:10
  Saved: schedule_WARANGAL-I_2026-01-30.xlsx

Saved consolidated: consolidated_schedule_2026-01-30.xlsx

SCHEDULING COMPLETE


## 9. View Results

In [13]:
# =============================================================================
# VIEW CHANGES ONLY
# =============================================================================

if len(consolidated_schedule) > 0:
    changes_only = consolidated_schedule[
        consolidated_schedule['action'] != 'NO_CHANGE'
    ].copy()
    
    print(f"Total changes across all depots: {len(changes_only)}")
    print("\nChanges by depot and action:")
    print(changes_only.groupby(['depot', 'action']).size().unstack(fill_value=0))
    
    print("\nAll changes:")
    display(changes_only)
else:
    print("No schedule data generated")

Total changes across all depots: 10

Changes by depot and action:
action      STOP
depot           
WARANGAL-I    10

All changes:


,service_number,route,product,dep_time,is_peak,is_protected,planned_trips,suggested_trips,trip_change,km_per_trip,planned_kms_day,suggested_kms_day,kms_change,avg_or_last_7d,action,reason,depot
0,NE09,NIZAMBAD,PE,03:45:00,False,False,2,0,-2,237.000000,474.0,0.0,-474.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
1,NZD1,NIZAMBAD,SL,03:35:00,False,False,2,0,-2,237.000000,474.0,0.0,-474.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
2,M112,BYRMBHEL,HT,13:00:00,False,False,3,0,-3,233.333333,700.0,0.0,-700.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
3,KKP5,BYRMBHEL,HT,15:30:00,False,False,3,0,-3,233.333333,700.0,0.0,-700.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
4,BYH1,MNGR-KPHB,HT,14:30:00,False,False,3,0,-3,238.666667,716.0,0.0,-716.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
5,KKP4,MNGR-KPHB,HT,15:00:00,False,False,3,0,-3,238.666667,716.0,0.0,-716.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
6,SLM2,SRSLM,HT,10:30:00,False,False,2,0,-2,365.000000,730.0,0.0,-730.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
7,VSP1,HNK-VSPT,HL,15:00:00,False,False,2,0,-2,663.000000,1326.0,0.0,-1326.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
8,CTR1,TIRUPATI,MB,13:30:00,False,False,2,0,-2,711.000000,1422.0,0.0,-1422.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I
9,BLD1,BNGL,HT,13:30:00,False,False,2,0,-2,746.000000,1492.0,0.0,-1492.0,0.0,STOP,"Stop service, very low OR=0.00",WARANGAL-I


In [14]:
# =============================================================================
# SUMMARY TABLE
# =============================================================================

if all_summaries:
    summary_rows = []
    for depot, summary in all_summaries.items():
        summary_rows.append({
            'Depot': depot,
            'Predicted PKM': summary['predicted_depot_pkm'],
            'Planned KMs': summary['depot_planned_kms'],
            'Target KMs': summary['depot_target_kms'],
            'Delta KMs': summary['delta_kms'],
            'KMs Change': summary['total_kms_change'],
            'Increases': summary['count_increase'],
            'Decreases': summary['count_decrease'],
            'Stops': summary['count_stop'],
            'Total Changed': summary['total_services_changed'],
        })
    
    summary_df = pd.DataFrame(summary_rows)
    
    print("SCHEDULING SUMMARY BY DEPOT")
    print("=" * 100)
    display(summary_df)
    
    print("\nTOTALS:")
    print(f"  Total Predicted PKM: {summary_df['Predicted PKM'].sum():,.0f}")
    print(f"  Total Planned KMs: {summary_df['Planned KMs'].sum():,.0f}")
    print(f"  Total Target KMs: {summary_df['Target KMs'].sum():,.0f}")
    print(f"  Total KMs Change: {summary_df['KMs Change'].sum():+,.0f}")
    print(f"  Total Services Changed: {summary_df['Total Changed'].sum()}")

SCHEDULING SUMMARY BY DEPOT


,Depot,Predicted PKM,Planned KMs,Target KMs,Delta KMs,KMs Change,Increases,Decreases,Stops,Total Changed
0,WARANGAL-I,2.061263e+06,69696.0,61074.449219,-8621.550781,-8750.0,0,0,10,10



TOTALS:
  Total Predicted PKM: 2,061,263
  Total Planned KMs: 69,696
  Total Target KMs: 61,074
  Total KMs Change: -8,750
  Total Services Changed: 10


In [15]:
# =============================================================================
# QA CHECKS
# =============================================================================

if len(consolidated_schedule) > 0:
    # Check 1: Protected services should never be cut
    bad_protected = consolidated_schedule[
        (consolidated_schedule['is_protected']) & 
        (consolidated_schedule['action'].isin(['DECREASE', 'STOP']))
    ]
    print(f"Protected services cut/stopped: {len(bad_protected)}")
    if len(bad_protected) > 0:
        display(bad_protected)
    
    # Check 2: Peak services that were cut
    peak_cut = consolidated_schedule[
        (consolidated_schedule['is_peak']) & 
        (consolidated_schedule['action'].isin(['DECREASE', 'STOP']))
    ]
    print(f"\nPeak services cut/stopped: {len(peak_cut)}")
    if len(peak_cut) > 0:
        display(peak_cut)
    
    print("\nQA checks complete")

Protected services cut/stopped: 0

Peak services cut/stopped: 0

QA checks complete


## 10. Summary

### Daily Workflow
```
1. Run demand_prediction.ipynb
   → Generates predictions in output/predictions/daily_predictions.parquet

2. Run supply_scheduling.ipynb (this notebook)
   → Reads predictions
   → Generates schedules in output/dynamic_schedule/{date}/

3. Run data_pipeline.ipynb (after day ends)
   → Updates actuals
   → Archives processed files
```

### Output Files
| File | Description |
|------|-------------|
| schedule_{depot}_{date}.xlsx | Suggested schedule per depot |
| summary_{depot}_{date}.json | Metrics and summary |
| consolidated_schedule_{date}.xlsx | All depots combined |

### Schedule Columns
| Column | Description |
|--------|-------------|
| service_number | Service identifier |
| planned_trips | Current scheduled trips |
| suggested_trips | Recommended trips |
| trip_change | Difference (+/-) |
| action | INCREASE / DECREASE / STOP / NO_CHANGE |
| reason | Explanation for the action |